# 05 Baseline Tweedie GLM

**Purpose:** first model for all three lines, with the validation and checks fixed in DECISIONS.md
(Oct 18 modelling plan): train on 2021–2023, validate on 2024, Tweedie deviance (power 1.5).
Then refit on 2021–2024 and write the first submission file.

Target and weight come from `build_model_target` (2025 level; weight = exposure × completeness).
Features come from `build_features` (excluded columns and reasons are listed in `src/features.py`).

In [1]:
import sys, time
sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
from src.target import DATA_DIR, LINES, load_policies, load_claims, build_model_target
from src.features import build_features
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

claims = load_claims()
SEGMENTS = {  # columns to check A/E by on validation
    "motor": ["driver_age", "vehicle_value_group", "region", "new_business", "cover_type", "telematics_opt_in"],
    "buildings": ["policyholder_age", "sum_insured", "region", "flood_zone", "soil_type", "dry_summer_index"],
    "contents": ["policyholder_age", "sum_insured", "region", "tenure", "security_level", "rainfall_mm"],
}

## 1. Build the modelling table for each line
**Purpose:** join the 2025-level target and weight to the features; check row counts and missing values.

In [2]:
data = {}
for line in LINES:
    p = load_policies(line, "train")
    mt, comp, loading = build_model_target(claims, p, line)
    F = build_features(line, "train", policies=p)
    df = mt.merge(F, on="row_id", how="left", validate="one_to_one")
    assert len(df) == len(p)
    data[line] = {"df": df, "loading": loading}
    feats = [c for c in F.columns if c != "row_id"]
    miss = df[feats].isna().mean()
    print(f"{line}: {len(df):,} rows, {len(feats)} features, loading {loading:.1f}; "
          f"features with missing values: {miss[miss > 0].round(3).to_dict()}")

motor: 979,968 rows, 58 features, loading 16.0; features with missing values: {'vehicle_age': 0.025, 'engine_size_cc': 0.025, 'annual_mileage': 0.025, 'overnight_parking': 0.025, 'telematics_score': 0.876, 'occupation': 0.122, 'marital_status': 0.123, 'household_size': 0.122, 'credit_band': 0.1, 'prior_claims_5y': 0.1, 'licence_points': 0.1, 'years_at_address': 0.123, 'previous_insurer': 0.1, 'flood_zone': 0.029, 'distance_to_river_km': 0.025, 'soil_type': 0.025, 'roof_type': 0.025, 'energy_rating': 0.025, 'customer_tenure': 0.1}
buildings: 707,726 rows, 48 features, loading 2.3; features with missing values: {'days_unoccupied': 0.025, 'smoke_alarms': 0.025, 'occupation': 0.025, 'marital_status': 0.025, 'household_size': 0.025, 'years_at_address': 0.025, 'flood_zone': 0.029, 'distance_to_river_km': 0.025, 'soil_type': 0.025, 'roof_type': 0.025, 'energy_rating': 0.025}
contents: 721,617 rows, 49 features, loading 0.0; features with missing values: {'security_level': 0.025, 'high_value_i

## 2. Validation: train 2021–2023, predict 2024
**Purpose:** the main model check. Predicted cost per row = predicted rate × weight, compared with the
actual 2025-level cost of 2024 policies. Segment flags show where A/E is outside 0.9–1.1.

In [3]:
results, valid_rows = {}, []
for line in LINES:
    df = data[line]["df"]
    feats = [c for c in df.columns if c not in ("row_id", "year", "exposure_clean", "target", "weight")]
    tr, va = df[df.year <= 2023], df[df.year == 2024]
    t0 = time.time()
    glm = BandedTweedieGLM().fit(tr[feats], tr.target / tr.weight, tr.weight)
    pred = glm.predict_rate(va[feats]) * va.weight.to_numpy()
    print(f"\n[{line}] fitted in {time.time() - t0:.0f}s")
    results[line] = evaluate(va.target, pred, va.weight, segments=va[SEGMENTS[line]], name=f"GLM {line} 2024")
    valid_rows.append(pd.DataFrame({"actual": va.target.to_numpy(), "pred": pred, "weight": va.weight.to_numpy()}))


[motor] fitted in 26s
===== GLM motor 2024: 254,698 rows =====
D2 (Tweedie 1.5): 0.0517 | overall A/E: 1.025 | top/bottom decile actual rate: 6.1x
      pred_rate  actual_rate    A/E
band                               
1       187.697      230.550  1.228
2       257.781      276.385  1.072
3       307.559      346.996  1.128
4       355.963      347.897  0.977
5       408.000      389.833  0.955
6       467.499      503.336  1.077
7       542.517      541.974  0.999
8       647.144      668.379  1.033
9       823.637      829.772  1.007
10     1414.447     1402.846  0.992

[segment flag] region: A/E outside 0.9-1.1
     share_of_weight    A/E
seg                        
R03            0.096  1.114
R06            0.043  1.206
R11            0.085  1.122

[segment flag] cover_type: A/E outside 0.9-1.1
                        share_of_weight    A/E
seg                                           
third_party_fire_theft            0.093  0.838

[buildings] fitted in 15s
===== GLM buildings 

In [4]:
# All lines together (closest to how Kaggle scores one file)
v = pd.concat(valid_rows, ignore_index=True)
overall = evaluate(v.actual, v.pred, v.weight, name="GLM all lines 2024", show=False)
summary = pd.DataFrame({k: {"D2": r["D2"], "A/E": r["A/E"], "top/bottom": r["top/bottom"]}
                        for k, r in {**results, "all lines": overall}.items()}).T
print(summary.round(4))

               D2     A/E  top/bottom
motor      0.0517  1.0246      6.0848
buildings  0.0577  0.9332      9.4076
contents   0.0623  1.0177      8.6123
all lines  0.1053  0.9860     29.7541


## 3. Refit on 2021–2024 and predict 2025
**Purpose:** use all years for the submission. Row prediction = (predicted rate + large-loss loading) × test exposure.

In [5]:
subs, models = [], {}
for line in LINES:
    df = data[line]["df"]
    feats = [c for c in df.columns if c not in ("row_id", "year", "exposure_clean", "target", "weight")]
    glm = BandedTweedieGLM().fit(df[feats], df.target / df.weight, df.weight)
    models[line] = glm
    pt = load_policies(line, "test")
    Ft = build_features(line, "test", policies=pt)
    rate = glm.predict_rate(Ft[feats])
    pred = np.maximum((rate + data[line]["loading"]) * pt.exposure_clean.to_numpy(), 0)
    subs.append(pd.DataFrame({"row_id": Ft.row_id, "prediction": pred}))
    exp_level = df.target.sum() / df.weight.sum() + data[line]["loading"]
    print(f"{line}: mean predicted rate {rate.mean() + data[line]['loading']:.1f} "
          f"(training level incl. loading {exp_level:.1f}); {len(pt):,} test rows")

motor: mean predicted rate 527.2 (training level incl. loading 563.9); 254,986 test rows
buildings: mean predicted rate 491.8 (training level incl. loading 466.6); 184,155 test rows
contents: mean predicted rate 111.2 (training level incl. loading 106.1); 187,498 test rows


In [6]:
sub = pd.concat(subs, ignore_index=True)
sample = pd.read_parquet(DATA_DIR / "sample_submission.parquet")
assert len(sub) == len(sample) and set(sub.row_id) == set(sample.row_id), "row_ids must match sample_submission"
assert sub.prediction.notna().all() and (sub.prediction >= 0).all()
out = Path("../submissions"); out.mkdir(exist_ok=True)
sub.to_csv(out / "glm_baseline.csv", index=False)
print(f"wrote {len(sub):,} rows to submissions/glm_baseline.csv")
print(sub.prediction.describe().round(1))

wrote 626,639 rows to submissions/glm_baseline.csv
count    626639.0
mean        307.0
std         343.4
min           0.0
25%          79.5
50%         221.7
75%         410.8
max       11497.9
Name: prediction, dtype: float64


## 4. Relativities (for review)
**Purpose:** check that the main effects make actuarial sense (sign and size), e.g. young drivers higher.

In [7]:
rel = models["motor"].relativities()
for f in ["driver_age", "ncb_years", "vehicle_value_group", "credit_band", "region"]:
    print(rel[rel.feature == f].round(3).to_string(index=False), "\n")

   feature level  relativity
driver_age   b01       0.763
driver_age   b02       0.690
driver_age   b03       0.610
driver_age   b04       0.586
driver_age   b05       0.534
driver_age   b06       0.538
driver_age   b07       0.561
driver_age   b08       0.640
driver_age   b09       0.727 

  feature level  relativity
ncb_years   b01       0.828
ncb_years   b02       0.722
ncb_years   b03       0.664
ncb_years   b04       0.568
ncb_years   b05       0.491
ncb_years   b06       0.466
ncb_years   b07       0.446
ncb_years   b08       0.439
ncb_years   b09       0.454 

            feature level  relativity
vehicle_value_group   b01       1.134
vehicle_value_group   b02       1.095
vehicle_value_group   b03       1.127
vehicle_value_group   b04       1.139
vehicle_value_group   b05       1.223
vehicle_value_group   b06       1.221
vehicle_value_group   b07       1.256
vehicle_value_group   b08       1.268
vehicle_value_group   b09       1.384 

    feature   level  relativity
credit_band 

In [2]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from src.target import LINES, load_policies, load_claims, build_model_target
from src.features import build_features
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

claims = load_claims()
SEGMENTS = {
    "motor": ["driver_age", "vehicle_value_group", "region", "new_business", "cover_type", "telematics_opt_in"],
    "buildings": ["policyholder_age", "sum_insured", "region", "flood_zone", "soil_type", "dry_summer_index"],
    "contents": ["policyholder_age", "sum_insured", "region", "tenure", "security_level", "rainfall_mm"],
}

# Modelling tables (same as section 1)
data = {}
for line in LINES:
    p = load_policies(line, "train")
    mt, comp, loading = build_model_target(claims, p, line)
    F = build_features(line, "train", policies=p)
    data[line] = {"df": mt.merge(F, on="row_id", how="left", validate="one_to_one"), "loading": loading}
    print(f"{line}: table ready")

# Two validation folds
FOLDS = [(2023, [2021, 2022]), (2024, [2021, 2022, 2023])]
rows, keep = [], {}
for line in LINES:
    df = data[line]["df"]
    feats = [c for c in df.columns if c not in ("row_id", "year", "exposure_clean", "target", "weight")]
    for vy, tys in FOLDS:
        tr, va = df[df.year.isin(tys)], df[df.year == vy]
        glm = BandedTweedieGLM().fit(tr[feats], tr.target / tr.weight, tr.weight)
        pred = glm.predict_rate(va[feats]) * va.weight.to_numpy()
        r = evaluate(va.target, pred, va.weight, segments=va[SEGMENTS[line]], name=f"{line} {vy}", show=False)
        lf = r["lift"]["A/E"]
        rows.append({"line": line, "valid_year": vy, "D2": r["D2"], "A/E": r["A/E"],
                     "A/E lowest decile": lf.iloc[0], "A/E top decile": lf.iloc[-1]})
        keep[(line, vy)] = r
print(pd.DataFrame(rows).round(3).to_string(index=False))

# Buildings: is the over-prediction in high-risk homes also there in mature 2023?
for vy in [2023, 2024]:
    r = keep[("buildings", vy)]
    print(f"\n[buildings {vy}] lift A/E by decile:", r["lift"]["A/E"].round(2).tolist())
    print(r["segments"]["dry_summer_index"].round(3).to_string())

motor: table ready
buildings: table ready
contents: table ready


/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [26] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)


     line  valid_year    D2   A/E  A/E lowest decile  A/E top decile
    motor        2023 0.048 1.044              1.206           0.945
    motor        2024 0.052 1.025              1.228           0.992
buildings        2023 0.052 1.006              1.172           0.938
buildings        2024 0.058 0.933              1.184           0.834
 contents        2023 0.058 1.069              1.546           1.049
 contents        2024 0.062 1.018              1.510           0.874

[buildings 2023] lift A/E by decile: [1.17, 1.65, 1.28, 1.5, 1.0, 0.92, 1.08, 0.84, 0.84, 0.94]
                 share_of_weight    A/E
seg                                    
(-0.001, 0.293]            0.201  1.158
(0.293, 0.477]             0.200  1.119
(0.477, 0.697]             0.199  0.886
(0.697, 0.822]             0.200  0.941
(0.822, 1.0]               0.200  0.962

[buildings 2024] lift A/E by decile: [1.18, 1.17, 1.23, 1.14, 1.2, 0.67, 0.98, 1.03, 0.79, 0.83]
                 share_of_weight    A/E
se